In [1]:
import pandas as pd
import regex as re
import numpy as np

In [2]:
def safe_equal(a, b):
    """Return True if values are equal, handling lists, Series, NaN, etc."""
    if isinstance(a, float) and isinstance(b, float):
        if np.isnan(a) and np.isnan(b):
            return True

    if isinstance(a, pd.Series):
        a = a.tolist()
    if isinstance(b, pd.Series):
        b = b.tolist()

    if hasattr(a, "tolist") and not isinstance(a, str):
        a = a.tolist()
    if hasattr(b, "tolist") and not isinstance(b, str):
        b = b.tolist()

    return a == b


def df_diff(before, after):
    """Full diff between two dataframes, safe for explode, Series cells, lists, etc."""
    shared_cols = before.columns.intersection(after.columns)

    removed_idx = before.index.difference(after.index)
    added_idx = after.index.difference(before.index)

    removed_rows = before.loc[removed_idx] if len(removed_idx) else pd.DataFrame()
    added_rows = after.loc[added_idx] if len(added_idx) else pd.DataFrame()

    common_idx = before.index.intersection(after.index)

    before_common = before.loc[common_idx, shared_cols]
    after_common = after.loc[common_idx, shared_cols]

    changes = []

    for idx in common_idx:
        row_before = before_common.loc[idx]
        row_after = after_common.loc[idx]

        for col in shared_cols:
            v1 = row_before[col]
            v2 = row_after[col]

            if not safe_equal(v1, v2):
                changes.append({
                    "index": idx,
                    "column": col,
                    "before": v1,
                    "after": v2,
                    "_change": "cell_changed"
                })

    changes = pd.DataFrame(changes)

    return {
        "added_rows": added_rows.assign(_change="row_added"),
        "removed_rows": removed_rows.assign(_change="row_removed"),
        "cell_changes": changes
    }


# --- Tracking wrapper ---
ops = 0
last_snapshot = None

def track(df, label):
    global ops, last_snapshot

    if last_snapshot is None:
        last_snapshot = df.copy(deep=True)
        print(f"[INIT SNAPSHOT] {label}")
        return

    diff = df_diff(last_snapshot, df)

    print(f"\n=== DIFF AFTER: {label} ===")
    print("Added rows:", len(diff["added_rows"]))
    print("Removed rows:", len(diff["removed_rows"]))
    print("Cell changes:", len(diff["cell_changes"]))

    ops += len(diff["added_rows"]) + len(diff["removed_rows"]) + len(diff["cell_changes"])
    last_snapshot = df.copy(deep=True)


In [3]:
data = pd.read_csv("data/processed_translation_data_4.csv")
track(data, "load data")

[INIT SNAPSHOT] load data


In [4]:
data["author"] = data["author"].str.split(':')
#track(data, "author split")
data = data.explode("author")
#track(data, "author explode")
data["author"] = data["author"].str.strip()
#track(data, "author strip")


=== DIFF AFTER: author split ===
Added rows: 0
Removed rows: 0
Cell changes: 115286


KeyboardInterrupt: 

In [ ]:
correct_colindex = ["author", "translator", "editor", "fore_afterword_author", 
                    "title", "title_original", "genre", "source_language", "source_literature",
                   "publisher", "publication_location", "publication_year", "physical_description",
                   "series", "publication_type", "target_audience", "edition", "n_pages", "content",
                   "issue", "notes"]

In [ ]:
for col in data.columns:
    if col not in correct_colindex:
        print(f"{col} is not in data.columns")

In [ ]:
data = data.drop(["Unnamed: 0", "id", "author_of_translated_book_under_review_to_be_removed"], axis=1)
track(data, "drop unwanted columns")

In [ ]:
data = data.loc[:, correct_colindex]
track(data, "reorder columns")

In [ ]:
#data.head(30).to_csv("data/demo-30.csv", index=False)

In [ ]:
# Made with the help of ChatGPT (unfortunately)
def split_name_year(series):
    """Split into name and year parts."""
    return series.str.split(r"(\d.*)", n=1, expand=True)[[0,1]]

def clean_text(series):
    return series.str.strip(" \t,")

def split_years(series):
    return series.str.split(r"-", n=1, expand=True)

def clean_year(series):
    return (
        series
        .astype(str)
        .str.replace(r"[^\d\-]", "", regex=True)
        .str.strip()
        .replace("", np.nan)
    )


In [ ]:
# --- Reset tracking for persons_df ---
last_snapshot = None
ops_persons = 0

def track_persons(df, label):
    global ops_persons, last_snapshot

    if last_snapshot is None:
        last_snapshot = df.copy(deep=True)
        print(f"[INIT SNAPSHOT persons_df] {label}")
        return

    #diff = df_diff(last_snapshot, df)

    print(f"\n=== DIFF AFTER (persons_df): {label} ===")
    print("Added rows:", len(diff["added_rows"]))
    print("Removed rows:", len(diff["removed_rows"]))
    print("Cell changes:", len(diff["cell_changes"]))

    #ops_persons += len(diff["added_rows"]) + len(diff["removed_rows"]) + len(diff["cell_changes"])
    last_snapshot = df.copy(deep=True)


In [ ]:
# Made with the help of ChatGPT (unfortunately)
# --- Build persons dataframe ---
persons = pd.concat([data["author"], data["translator"], data["editor"], data["fore_afterword_author"]]).dropna().drop_duplicates()

persons_df = pd.DataFrame()
persons_df[["name", "year"]] = split_name_year(persons)
#track_persons(persons_df, "initial persons_df")
persons_df = persons_df.reset_index(drop=True)
#track_persons(persons_df, "reset index")

# Fix known problematic cases
replacements = {
    "497/496-406/405 eKr: Demokritos u 460- u 370 eKr": "497-406",
    "497/496-406/405 eKr": "497-406",
    "1213(1219)-1292": "1213-1292"
}
persons_df["year"] = persons_df["year"].replace(replacements)
#track_persons(persons_df, "replace problematic years")

# Split birth/death years using broader separators
persons_df[["birth_year", "death_year"]] = persons_df["year"].str.split(
    r"[\/\-\.—–\(\)]", n=1, expand=True
)
#track_persons(persons_df, "split birth/death")

# Clean values
persons_df["birth_year"] = clean_year(persons_df["birth_year"])
persons_df["death_year"] = clean_year(persons_df["death_year"])
#track_persons(persons_df, "clean years")

# Convert to numeric
persons_df["birth_year"] = pd.to_numeric(persons_df["birth_year"], errors="coerce")
persons_df["death_year"] = pd.to_numeric(persons_df["death_year"], errors="coerce")
#track_persons(persons_df, "convert to numeric")

# Drop intermediate column
persons_df = persons_df.drop(columns=["year"])
#track_persons(persons_df, "drop year column")

# --- Done ---
print(persons_df.head())

In [ ]:
persons_df.name = persons_df.name.str.strip()

In [ ]:
persons_df.columns

In [ ]:
pd.set_option("display.max_columns", 100)
#pd.DataFrame(persons_df.name).head(60)

In [ ]:
# Created with the help of Microsoft Copilot
def split_name(name):
    if pd.isna(name):
        return pd.Series([np.nan, np.nan], index=["first_name", "last_name"])

    name = name.strip()

    # Case 1: comma present → "Lastname, Firstname"
    if "," in name:
        last, first = name.split(",", 1)
        last = last.strip()
        first = first.strip()

        # If first part is empty (e.g., "Zeltmatis,")
        if first == "":
            first = np.nan

        return pd.Series([first, last], index=["first_name", "last_name"])

    # Case 2: no comma → split at last space
    if " " in name:
        left, right = name.rsplit(" ", 1)
        left = left.strip()
        right = right.strip()

        # If right part is empty, treat as no split
        if right == "":
            return pd.Series([np.nan, name], index=["first_name", "last_name"])

        return pd.Series([left, right], index=["first_name", "last_name"])

    # Case 3: single word → lastname only
    return pd.Series([np.nan, name], index=["first_name", "last_name"])

In [ ]:
persons_df[["first_name", "last_name"]] = persons_df["name"].apply(split_name)
#track_persons(persons_df, "split names")

In [ ]:
persons_df

In [ ]:
persons_df = persons_df.drop("name", axis=1)
track_persons(persons_df, "drop name")
persons_df = persons_df.loc[:, ["first_name", "last_name", "birth_year", "death_year"]]
track_persons(persons_df, "reorder columns")

In [ ]:
persons_df

In [ ]:
persons_df = persons_df.drop_duplicates()
track_persons(persons_df, "drop duplicates")
print("\nTOTAL OPERATIONS:", ops)

In [ ]:
persons_df.to_csv("data/clean_for_sql/persons.csv", index=False, quoting=False)

In [ ]:
data.genre.unique()

In [ ]:
# Created with the help of Copilot
def make_table(df, column_name, new_name=None):
    if new_name is None:
        new_name = column_name
    return (
        df[[column_name]]
        .dropna()
        .drop_duplicates()
        .sort_values(column_name)
        .reset_index(drop=True)
        .rename(columns={column_name: new_name})
    )

In [ ]:
data.source_language = data.source_language.str.strip(": ")
data.source_literature = data.source_literature.str.strip(": ")

In [ ]:
genres              = make_table(data, "genre", "est")
literatures         = make_table(data, "source_literature", "est")
publishers          = make_table(data, "publisher", "name")
locations           = make_table(data, "publication_location", "name")
publication_types   = make_table(data, "publication_type", "est")
languages           = make_table(data, "source_language", "est")
series              = make_table(data, "series", "name")
target_audiences    = make_table(data, "target_audience", "est")

In [ ]:
genres

In [ ]:
genres.to_csv("data/clean_for_sql/genres.csv", index=False, quoting=False)
literatures.to_csv("data/clean_for_sql/literatures.csv", index=False, quoting=False)
publishers.to_csv("data/clean_for_sql/publishers.csv", index=False, quoting=False)
locations.to_csv("data/clean_for_sql/locations.csv", index=False, quoting=False)
publication_types.to_csv("data/clean_for_sql/publication_types.csv", index=False, quoting=False)
languages.to_csv("data/clean_for_sql/languages.csv", index=False, quoting=False)
series.to_csv("data/clean_for_sql/series.csv", index=False, quoting=False)
target_audiences.to_csv("data/clean_for_sql/target_audiences.csv", index=False, quoting=False)

In [ ]:
data

In [ ]:
data.to_csv("data/clean_for_sql/translations.csv", index=False, quoting=False)

In [ ]:
"""TOTAL OPERATIONS: 140273"""